# Data audit and cleaning 


In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / 'data' / 'raw' / 'Charging_Data.csv').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find data/raw/Charging_Data.csv from the current directory.')
sys.path.insert(0, str(PROJECT_ROOT / 'team' / 'mariam_availability'))


## 1. Raw file audit

In [13]:
raw_path = PROJECT_ROOT / 'data' / 'raw' / 'Charging_Data.csv'
raw = pd.read_csv(raw_path, encoding='cp1252')
print(raw.shape)
raw.dtypes

(441077, 24)


UserID                       int64
Charging Post ID             int64
Location Information           str
District Name                  str
Order creation time            str
Transaction power/kwh      float64
Electricity cost/Yuan      float64
Service charge/Yuan        float64
Transaction Amount/Yuan    float64
Actual Payment/Yuan        float64
Start Time                     str
End Time                       str
Payment time                   str
Temperature(¡æ)            float64
Relative Humidity(%)         int64
Precipitation(mm)          float64
end_cause                      str
is_full_charge               int64
is_user_stop                 int64
is_nonsystem_fault           int64
is_system_issue              int64
is_EV_fault                  int64
is_other_fault               int64
is_abnormal                  int64
dtype: object

In [14]:
raw.isna().sum()

UserID                     0
Charging Post ID           0
Location Information       0
District Name              0
Order creation time        0
Transaction power/kwh      0
Electricity cost/Yuan      0
Service charge/Yuan        0
Transaction Amount/Yuan    0
Actual Payment/Yuan        0
Start Time                 0
End Time                   0
Payment time               0
Temperature(¡æ)            0
Relative Humidity(%)       0
Precipitation(mm)          0
end_cause                  0
is_full_charge             0
is_user_stop               0
is_nonsystem_fault         0
is_system_issue            0
is_EV_fault                0
is_other_fault             0
is_abnormal                0
dtype: int64

In [15]:
raw.head()

,UserID,Charging Post ID,Location Information,District Name,Order creation time,Transaction power/kwh,Electricity cost/Yuan,Service charge/Yuan,Transaction Amount/Yuan,Actual Payment/Yuan,...,Relative Humidity(%),Precipitation(mm),end_cause,is_full_charge,is_user_stop,is_nonsystem_fault,is_system_issue,is_EV_fault,is_other_fault,is_abnormal
0,1,1,Tourist Attraction,Nanhu,2020-12-31 23:35:55\t,6.24,2.60,3.74,6.34,6.34,...,45,0.0,User stops charging,0,1,0,0,0,0,0
1,2,2,Tourist Attraction,Nanhu,2020-12-31 23:35:17\t,0.62,0.26,0.37,0.63,0.63,...,45,0.0,User stops charging,0,1,0,0,0,0,0
2,3,3,Tourist Attraction,Nanhu,2020-12-31 22:01:34\t,28.01,11.66,16.81,28.47,28.47,...,45,0.0,User stops charging,0,1,0,0,0,0,0
3,4,2,Tourist Attraction,Nanhu,2020-12-31 22:21:37\t,5.20,2.17,3.12,5.29,5.29,...,45,0.0,User stops charging,0,1,0,0,0,0,0
4,5,1,Tourist Attraction,Nanhu,2020-12-31 22:10:30\t,4.58,1.91,2.75,4.66,4.66,...,45,0.0,Charging ends normally,1,0,0,0,0,0,0


In [16]:
print(repr(raw['Start Time'].iloc[0]))
print(raw['Location Information'].unique())

'2020-12-31 23:35:43\t'
<StringArray>
[           'Tourist Attraction',     'Financial Industrial Park',
               'Technology Park',                 'Shopping Mall',
                        'Park A',             'Industrial Park\t',
                        'Park B',                   'Bus Station',
             'Government Agency', 'Expressway Service District B',
 'Expressway Service District A', 'Expressway Service District C',
              'Wholesale Market']
Length: 13, dtype: str


## 2. Cardinality checks
Each charger should belong to exactly one location and district.

In [17]:
g = raw.groupby('Charging Post ID').agg(n=('UserID','size'),
        locations=('Location Information','nunique'), districts=('District Name','nunique'))
print('chargers:', len(g), '| max locations per charger:', g['locations'].max(), '| max districts:', g['districts'].max())
raw.groupby(['District Name','Location Information'])['Charging Post ID'].nunique()

chargers: 92 | max locations per charger: 1 | max districts: 1


District Name  Location Information         
Nanhu          Financial Industrial Park        8
               Technology Park                  4
               Tourist Attraction               8
Tongxiang      Bus Station                      8
               Industrial Park\t                8
               Park A                           8
               Park B                           8
               Shopping Mall                    8
Xiuzhou        Expressway Service District A    4
               Expressway Service District B    4
               Expressway Service District C    8
               Government Agency                8
               Wholesale Market                 8
Name: Charging Post ID, dtype: int64

## 3. Run the cleaner

In [ ]:
# Clean the raw data directly in this notebook.
df = raw.copy()
df.index.name = 'raw_row_index'

# Remove trailing spaces and tabs from text fields.
for column in df.select_dtypes(include=['object', 'string']).columns:
    df[column] = df[column].str.strip()

# Standardize the columns used by the availability analysis.
df = df.rename(columns={
    column: 'temp_c' for column in df.columns if column.startswith('Temperature')
})
df = df.rename(columns={
    'Charging Post ID': 'charger_id',
    'Location Information': 'location',
    'District Name': 'district',
    'Start Time': 'start',
    'End Time': 'end',
    'Transaction power/kwh': 'energy_kwh',
    'Relative Humidity(%)': 'humidity',
    'Precipitation(mm)': 'precip_mm',
})

# Parse dates and record rows removed at each cleaning step.
for column in ['start', 'end', 'Order creation time', 'Payment time']:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors='coerce')
invalid_timestamps = df.loc[df[['start', 'end']].isna().any(axis=1)].copy()
df = df.dropna(subset=['start', 'end']).copy()
df['duration_min'] = (df['end'] - df['start']).dt.total_seconds() / 60
nonpositive_sessions = df.loc[df['duration_min'] <= 0].copy()
df = df.loc[df['duration_min'] > 0].copy()
duplicate_sessions = df.loc[df.duplicated()].copy()
df = df.drop_duplicates()

# Keep the first session per charger and reject only later overlapping sessions.
sorted_sessions = df.sort_values(['charger_id', 'start'], kind='mergesort').copy()
prior_max_end = sorted_sessions.groupby('charger_id', sort=False)['end'].transform(
    lambda end: end.cummax().shift()
)
first_sessions = sorted_sessions.loc[prior_max_end.isna()].copy()
overlap_mask = prior_max_end.notna() & (sorted_sessions['start'] < prior_max_end)
overlap_sessions = sorted_sessions.loc[overlap_mask].copy()
overlap_sessions['latest_prior_end'] = prior_max_end.loc[overlap_mask]
overlap_sessions['overlap_minutes'] = (
    overlap_sessions['latest_prior_end'] - overlap_sessions['start']
).dt.total_seconds() / 60

# Retain the first session for each charger. Its own start/end are valid;
# only its prior session end is unavailable because no earlier session exists.
keep_mask = prior_max_end.isna() | sorted_sessions['start'].ge(prior_max_end)
df = sorted_sessions.loc[keep_mask].copy()

# Save the cleaned dataset and row-level cleaning evidence.
output_path = PROJECT_ROOT / 'data' / 'processed' / 'Charging_Data_clean.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f'Saved {len(df):,} cleaned rows to {output_path}')
df.head()

## 4. Validate the cleaned data

In [19]:
assert df['duration_min'].gt(0).all()
assert not df.duplicated().any()
prev_end = df.groupby('charger_id')['end'].shift()
assert not (df['start'] < prev_end).any(), 'overlaps remain'
print('all checks passed')

all checks passed


In [ ]:
df['duration_min'].describe(percentiles=[.01,.05,.5,.95,.99]).round(1)

In [ ]:
TABLES = PROJECT_ROOT / 'team' / 'mariam_availability' / 'reports' / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)

excluded_parts = []
for excluded_rows, reason in [
    (invalid_timestamps, 'invalid_start_or_end'),
    (nonpositive_sessions, 'nonpositive_duration'),
    (duplicate_sessions, 'exact_duplicate'),
    (overlap_sessions, 'overlap_with_prior_session'),
]:
    if not excluded_rows.empty:
        part = excluded_rows.copy()
        part['exclusion_reason'] = reason
        excluded_parts.append(part)
exclusions = pd.concat(excluded_parts).sort_index() if excluded_parts else pd.DataFrame()
exclusions['raw_data_row'] = exclusions.index + 2
for column in ['latest_prior_end', 'overlap_minutes']:
    if column not in exclusions:
        exclusions[column] = pd.NA
exclusion_columns = [
    'raw_data_row', 'charger_id', 'start', 'end', 'duration_min',
    'exclusion_reason', 'latest_prior_end', 'overlap_minutes',
]
for column in exclusion_columns:
    if column not in exclusions:
        exclusions[column] = pd.NA
exclusions[exclusion_columns].sort_values('raw_data_row').to_csv(
    TABLES / 'cleaning_exclusions.csv', index=False
)

# Record first sessions separately: their own session times are present, but no prior end exists.
first_session_rows = first_sessions.copy()
first_session_rows['raw_data_row'] = first_session_rows.index + 2
first_session_rows['reason'] = 'first session per charger; no prior session end exists'
first_session_rows[['raw_data_row', 'charger_id', 'start', 'end', 'duration_min', 'reason']].to_csv(
    TABLES / 'cleaning_first_session_rows_retained.csv', index=False
)
pre_overlap_count = len(sorted_sessions)
actual_overlap_count = len(overlap_sessions)
former_bug_count = pre_overlap_count - actual_overlap_count - len(first_sessions)
count_audit = pd.Series({
    'raw_rows': len(raw),
    'invalid_timestamp_rows_removed': len(invalid_timestamps),
    'nonpositive_duration_rows_removed': len(nonpositive_sessions),
    'exact_duplicate_rows_removed': len(duplicate_sessions),
    'valid_rows_before_overlap_filter': pre_overlap_count,
    'true_overlapping_sessions_removed': actual_overlap_count,
    'first_sessions_retained_with_missing_prior_end': len(first_sessions),
    'first_sessions_missing_start': int(first_sessions['start'].isna().sum()),
    'first_sessions_missing_end': int(first_sessions['end'].isna().sum()),
    'clean_rows_current_pipeline': len(df),
    'clean_rows_from_former_NaT_filter': former_bug_count,
    'count_difference_explained': len(first_sessions),
    'first_sessions_removed_for_missing_own_times': 0,
    'clean_rows_reported_by_user': 441064,
    'reported_count_matches_fixed_pipeline': len(df) == 441064,
}, name='value').to_frame()
count_audit.to_csv(TABLES / 'cleaning_count_reconciliation.csv')

session_audit = pd.Series({
    'raw_rows': len(raw),
    'clean_rows': len(df),
    'removed_rows': len(raw) - len(df),
    'invalid_timestamp_rows_removed': len(invalid_timestamps),
    'nonpositive_duration_rows_removed': len(nonpositive_sessions),
    'exact_duplicate_rows_removed': len(duplicate_sessions),
    'overlap_sessions_removed': len(overlap_sessions),
    'raw_missing_cells': int(raw.isna().sum().sum()),
    'clean_missing_cells': int(df.isna().sum().sum()),
    'sessions_over_12h': int((df['duration_min'] > 720).sum()),
    'duration_p50_min': float(df['duration_min'].quantile(.50)),
    'duration_p95_min': float(df['duration_min'].quantile(.95)),
    'duration_p99_min': float(df['duration_min'].quantile(.99)),
    'duration_max_min': float(df['duration_min'].max()),
}, name='value').to_frame()
session_audit.to_csv(TABLES / 'session_audit.csv')
raw_missing = raw.isna().sum().rename('missing_values').to_frame()
raw_missing['missing_pct'] = raw_missing['missing_values'] / len(raw) * 100
clean_missing = df.isna().sum().rename('missing_values').to_frame()
clean_missing['missing_pct'] = clean_missing['missing_values'] / len(df) * 100
raw_missing.to_csv(TABLES / 'missingness_raw.csv')
clean_missing.to_csv(TABLES / 'missingness_clean.csv')
count_audit

## 5. Columns forbidden as model inputs 
These are only known after the session ends. Keep them out of every availability feature set.

In [21]:
forbidden = ['end','Payment time','energy_kwh','Electricity cost/Yuan','Service charge/Yuan',
             'Transaction Amount/Yuan','Actual Payment/Yuan','end_cause','duration_min',
             'is_full_charge','is_user_stop','is_nonsystem_fault','is_system_issue','is_EV_fault','is_other_fault','is_abnormal']
allowed_base = ['charger_id','location','district','start','temp_c','humidity','precip_mm']
print('allowed base columns:', allowed_base)

allowed base columns: ['charger_id', 'location', 'district', 'start', 'temp_c', 'humidity', 'precip_mm']
